# 🤗 Notebook Complementar — Hugging Face e BERT
### Aula 2 — Tecnologias e Ferramentas para PLN

---

## 🎯 O que você vai aprender

- O que é o **BERT** e como ele revolucionou o PLN
- Como o **mecanismo de atenção** funciona internamente
- Como usar **pipelines** do Hugging Face para múltiplas tarefas
- Como **tokenizar** e inspecionar embeddings do BERT
- Diferenças entre **BERT, DistilBERT e RoBERTa**
- Como usar modelos em **Português**

---

## 📚 Teoria: O que é o BERT?

### A Revolução dos Transformers

Antes do BERT, os modelos de PLN processavam o texto **palavra por palavra** (esquerda → direita),  
como um leitor que nunca pode olhar para frente ou para trás ao mesmo tempo.

Em 2018, pesquisadores do Google publicaram o paper **"BERT: Pre-training of Deep Bidirectional  
Transformers for Language Understanding"**, que mudou tudo.

### BERT = Bidirectional Encoder Representations from Transformers

| Palavra | Significado |
|---|---|
| **B**idirectional | Lê o texto nos **dois sentidos** ao mesmo tempo (esquerda→direita E direita→esquerda) |
| **E**ncoder | Usa a parte **codificadora** da arquitetura Transformer |
| **R**epresentations | Gera **representações vetoriais** ricas para cada palavra |
| **T**ransformers | Baseado no mecanismo de **atenção** (Attention Is All You Need, 2017) |

### Por que Bidirecional é tão importante?

```
Frase: "Fui ao banco sacar dinheiro"

Modelo UNIDIRECIONAL (esquerda→direita):
  "banco" → só viu: Fui, ao
  Não sabe ainda o que vem depois!

Modelo BIDIRECIONAL (BERT):
  "banco" → vê: Fui, ao, sacar, dinheiro
  Contexto completo → banco FINANCEIRO ✅
```

### Como o BERT foi Treinado?

O BERT foi pré-treinado em dois objetivos simultâneos:

**1. Masked Language Model (MLM) — "Adivinhe a palavra mascarada"**
```
Original: "O gato sentou no tapete"
Input:    "O [MASK] sentou no tapete"
BERT aprende: [MASK] = gato (ou cachorro, animal...)
```

**2. Next Sentence Prediction (NSP) — "Esta sentença vem depois da outra?"**
```
Sentença A: "O céu está azul."
Sentença B: "A grama é verde." → IsNext: NÃO
Sentença B: "Não há nuvens."   → IsNext: SIM
```

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÃO INICIAL
# ─────────────────────────────────────────────────────────────
from transformers import (
    AutoTokenizer, AutoModel,
    BertTokenizer, BertModel,
    pipeline
)
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

MODELO_BERT = 'distilbert-base-uncased-finetuned-sst-2-english'
MODELO_BERT_PT = 'neuralmind/bert-base-portuguese-cased'

print('✅ Bibliotecas carregadas!')
print(f'   Transformers : ', end='')
import transformers; print(transformers.__version__)
print(f'   PyTorch      : {torch.__version__}')
print(f'   CUDA         : {torch.cuda.is_available()}')

---
## 📖 Parte 1 — O Tokenizador BERT em Profundidade

### Como o BERT divide as palavras?

O BERT usa **WordPiece tokenization** — divide palavras desconhecidas em subpalavras.  
Isso resolve o problema de palavras fora do vocabulário (OOV — Out Of Vocabulary).

### 1.1 Inspecionando a Tokenização

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOKENIZADOR BERT — WordPiece Tokenization
#
# O BERT divide palavras em subpalavras (wordpieces)
# Prefixo '##' indica continuação de uma palavra anterior
# ─────────────────────────────────────────────────────────────

print('🔄 Carregando tokenizador...')
tokenizador = AutoTokenizer.from_pretrained(MODELO_BERT)
print('✅ Tokenizador carregado!\n')

# Frases de exemplo — progressão de complexidade
frases_exemplo = [
    'NLP is fascinating',
    'The transformer architecture revolutionized language models',
    'Bidirectionality is the key innovation in BERT',
    'Supercalifragilisticexpialidocious is a long word',  # Palavra rara!
]

for frase in frases_exemplo:
    # Tokenizar (sem adicionar tokens especiais)
    tokens = tokenizador.tokenize(frase)

    # Codificar (com tokens especiais [CLS] e [SEP])
    ids = tokenizador.encode(frase)
    ids_sem_especiais = tokenizador.encode(frase, add_special_tokens=False)

    print(f'📝 "{frase}"')
    print(f'   Tokens     : {tokens}')
    print(f'   IDs (c/ especiais): {ids}')
    print(f'   Qtd tokens : {len(tokens)} (+ [CLS] e [SEP] = {len(ids)})')
    print()

print('─' * 60)
print('💡 Tokens especiais:')
print(f'   [CLS] = ID {tokenizador.cls_token_id}')
print(f'   [SEP] = ID {tokenizador.sep_token_id}')
print(f'   [UNK] = ID {tokenizador.unk_token_id}')
print(f'   [PAD] = ID {tokenizador.pad_token_id}')
print(f'   Vocabulário: {tokenizador.vocab_size:,} tokens')

---
### 1.2 Visualizando a Tokenização de Subpalavras

In [ ]:
# ─────────────────────────────────────────────────────────────
# VISUALIZAÇÃO DA TOKENIZAÇÃO
# Mostra como palavras complexas são divididas em subpalavras
# ─────────────────────────────────────────────────────────────

frase_viz = 'The transformer revolutionized natural language processing and understanding'
tokens = tokenizador.tokenize(frase_viz)
ids    = tokenizador.encode(frase_viz)

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# ── Gráfico 1: Tokens e IDs
ax1 = axes[0]
cores = ['#CECBF6' if not t.startswith('##') else '#FAEEDA'
         for t in ['[CLS]'] + tokens + ['[SEP]']]
todos_tokens = ['[CLS]'] + tokens + ['[SEP]']

for i, (tok, cor) in enumerate(zip(todos_tokens, cores)):
    ax1.add_patch(plt.Rectangle((i*1.1, 0), 1.0, 0.8,
                  facecolor=cor, edgecolor='#534AB7', linewidth=0.8))
    ax1.text(i*1.1 + 0.5, 0.55, tok, ha='center', va='center',
             fontsize=8, fontweight='bold', color='#26215C')
    ax1.text(i*1.1 + 0.5, 0.22, str(ids[i]), ha='center', va='center',
             fontsize=7, color='#5F5E5A')

ax1.set_xlim(-0.1, len(todos_tokens)*1.1 + 0.1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('Tokens BERT (roxo = palavra completa | laranja = subpalavra ##)',
              fontweight='bold', pad=8)

# ── Gráfico 2: Mapa de calor dos IDs
ax2 = axes[1]
ids_array = np.array(ids).reshape(1, -1)
im = ax2.imshow(ids_array, aspect='auto', cmap='RdPu', alpha=0.85)
for i, (tok, id_val) in enumerate(zip(todos_tokens, ids)):
    ax2.text(i, 0, f'{tok}\n{id_val}', ha='center', va='center',
             fontsize=7.5, fontweight='bold')
ax2.set_title('IDs numéricos correspondentes a cada token',
              fontweight='bold', pad=8)
ax2.axis('off')

plt.suptitle(f'Tokenização BERT: "{frase_viz[:55]}..."',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 📖 Parte 2 — BERT Embeddings: Representações Contextuais

### O que torna os embeddings do BERT especiais?

Modelos antigos (Word2Vec, GloVe) geravam um **único vetor** para cada palavra,  
independente do contexto. O BERT gera **vetores diferentes** para a mesma palavra  
dependendo do contexto em que ela aparece.

```
Word2Vec:  banco → [0.32, -0.18, 0.74]  (sempre o mesmo)

BERT:  "Fui ao banco sacar dinheiro"  → banco → [0.91, 0.23, -0.45]  (financeiro)
       "Sentei no banco da praça"      → banco → [0.12, 0.87, 0.34]  (móvel)
```

### 2.1 Extraindo Embeddings com BERT

In [ ]:
# ─────────────────────────────────────────────────────────────
# EMBEDDINGS CONTEXTUAIS DO BERT
# Demonstra que a mesma palavra tem vetores diferentes
# dependendo do contexto
# ─────────────────────────────────────────────────────────────

print('🔄 Carregando modelo BERT...')
tokenizador_bert = AutoTokenizer.from_pretrained('bert-base-uncased')
modelo_bert = AutoModel.from_pretrained('bert-base-uncased')
modelo_bert.eval()  # Modo de inferência (sem gradientes)
print('✅ Modelo carregado!\n')

def obter_embedding_palavra(frase, palavra_alvo):
    """
    Extrai o embedding contextual de uma palavra específica em uma frase.

    O embedding é a representação vetorial de 768 dimensões
    gerada pelo BERT para aquela palavra naquele contexto específico.
    """
    # Tokenizar a frase completa
    inputs = tokenizador_bert(frase, return_tensors='pt')
    tokens = tokenizador_bert.convert_ids_to_tokens(inputs['input_ids'][0])

    # Encontrar a posição da palavra alvo nos tokens
    idx_alvo = None
    for i, token in enumerate(tokens):
        if token == palavra_alvo.lower():
            idx_alvo = i
            break

    if idx_alvo is None:
        print(f'⚠️  "{palavra_alvo}" não encontrado como token único.')
        return None, tokens

    # Extrair embeddings sem calcular gradientes (mais eficiente)
    with torch.no_grad():
        outputs = modelo_bert(**inputs)

    # last_hidden_state: [batch, seq_len, 768]
    # Pegar o embedding da palavra alvo
    embedding = outputs.last_hidden_state[0, idx_alvo, :].numpy()
    return embedding, tokens

# Duas frases com a palavra 'bank' em contextos diferentes
frase_financeira = 'I went to the bank to withdraw money'
frase_rio        = 'We sat on the river bank watching the sunset'

emb_financeiro, tokens_fin = obter_embedding_palavra(frase_financeira, 'bank')
emb_rio,        tokens_rio = obter_embedding_palavra(frase_rio,        'bank')

print(f'📄 Frase 1: "{frase_financeira}"')
print(f'   Tokens  : {tokens_fin}')
print(f'   Embedding "bank" (5 primeiras dims): {emb_financeiro[:5].round(4)}')
print()
print(f'📄 Frase 2: "{frase_rio}"')
print(f'   Tokens  : {tokens_rio}')
print(f'   Embedding "bank" (5 primeiras dims): {emb_rio[:5].round(4)}')
print()

# Similaridade de cosseno entre os dois embeddings
# (1.0 = idênticos, 0.0 = perpendiculares)
similaridade = np.dot(emb_financeiro, emb_rio) / (
    np.linalg.norm(emb_financeiro) * np.linalg.norm(emb_rio)
)
print(f'📊 Similaridade de cosseno entre os dois embeddings de "bank": {similaridade:.4f}')
print()
if similaridade < 0.8:
    print('✅ Os embeddings são DIFERENTES → BERT capturou o contexto!')
else:
    print('ℹ️  Os embeddings são similares (modelo base pode variar por contexto)')

---
### 2.2 Visualizando Embeddings em 2D

Como vetores de 768 dimensões são impossíveis de visualizar diretamente,  
usamos redução de dimensionalidade para ver os embeddings em 2D.

In [ ]:
# ─────────────────────────────────────────────────────────────
# VISUALIZAÇÃO 2D DOS EMBEDDINGS
# PCA reduz 768 dimensões → 2 para visualização
# Palavras semanticamente próximas ficam próximas no gráfico
# ─────────────────────────────────────────────────────────────

from sklearn.decomposition import PCA

def obter_embedding_frase(frase):
    """Retorna o embedding da frase inteira (token [CLS])."""
    inputs = tokenizador_bert(frase, return_tensors='pt',
                              truncation=True, max_length=64)
    with torch.no_grad():
        outputs = modelo_bert(**inputs)
    # [CLS] token (índice 0) representa a frase inteira
    return outputs.last_hidden_state[0, 0, :].numpy()

# Frases agrupadas por categoria
grupos = {
    '🐾 Animais': [
        'The dog barked at the cat',
        'A wolf howled in the forest',
        'The lion hunted its prey',
        'A dolphin swam in the ocean',
    ],
    '💻 Tecnologia': [
        'The computer processed the data quickly',
        'Machine learning models need large datasets',
        'The neural network was trained on GPUs',
        'Python is a popular programming language',
    ],
    '🍕 Culinária': [
        'The pizza was baked in a wood oven',
        'She cooked a delicious pasta for dinner',
        'The soup was warm and comforting',
        'Fresh bread smells amazing in the morning',
    ],
}

# Extrair embeddings de todas as frases
frases_todas, labels_todas, grupos_todos = [], [], []
for grupo, frases in grupos.items():
    for frase in frases:
        frases_todas.append(frase)
        labels_todas.append(frase[:30] + '...')
        grupos_todos.append(grupo)

print('🔄 Extraindo embeddings...')
embeddings = np.array([obter_embedding_frase(f) for f in frases_todas])
print(f'✅ {len(embeddings)} embeddings de 768 dimensões extraídos')

# Redução PCA: 768 → 2 dimensões
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(embeddings)
print(f'📉 PCA: variância explicada pelas 2 componentes: '
      f'{pca.explained_variance_ratio_.sum():.1%}')

# Plotar
fig, ax = plt.subplots(figsize=(11, 7))
cores_grupos = {'🐾 Animais': '#1D9E75', '💻 Tecnologia': '#534AB7', '🍕 Culinária': '#D85A30'}

for i, (x, y) in enumerate(emb_2d):
    grupo = grupos_todos[i]
    cor = cores_grupos[grupo]
    ax.scatter(x, y, c=cor, s=120, alpha=0.85, edgecolors='white', linewidths=0.8, zorder=3)
    ax.annotate(frases_todas[i][:35], (x, y), fontsize=7.5,
                xytext=(5, 5), textcoords='offset points', color='#1A1A1A')

# Legenda manual
from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=g, edgecolor='white')
          for g, c in cores_grupos.items()]
ax.legend(handles=legend, loc='upper left', fontsize=10)
ax.set_title('Embeddings BERT em 2D (PCA)\nFrases semanticamente similares ficam próximas',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Componente Principal 1', fontsize=10)
ax.set_ylabel('Componente Principal 2', fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

---
## 📖 Parte 3 — Pipelines do Hugging Face

### 3.1 Análise de Sentimento Detalhada

In [ ]:
# ─────────────────────────────────────────────────────────────
# ANÁLISE DE SENTIMENTO COM SCORES VISUAIS
# ─────────────────────────────────────────────────────────────

sentimento = pipeline('sentiment-analysis',
                       model='distilbert-base-uncased-finetuned-sst-2-english')

reviews = [
    'This product is absolutely fantastic! Best purchase I ever made.',
    'Terrible quality. Broke after one day. Very disappointed.',
    'It works, nothing special. Average product for the price.',
    'Amazing customer service! They resolved my issue immediately.',
    'Do not buy this. Complete waste of money.',
    'Pretty good overall but the packaging could be better.',
]

resultados = sentimento(reviews)

print('📊 Análise de Sentimento — Reviews de Produtos\n')
fig, ax = plt.subplots(figsize=(12, 5))

labels_plot, scores_pos, scores_neg, cores_plot = [], [], [], []
for rev, res in zip(reviews, resultados):
    label = rev[:45] + '...'
    labels_plot.append(label)
    score = res['score']
    if res['label'] == 'POSITIVE':
        scores_pos.append(score)
        scores_neg.append(0)
        cores_plot.append('#1D9E75')
    else:
        scores_pos.append(0)
        scores_neg.append(score)
        cores_plot.append('#D85A30')

    emoji = '😊' if res['label'] == 'POSITIVE' else '😞'
    print(f'{emoji} {res["label"]:<10} ({score:.2%}) → "{rev[:50]}"')

y_pos = range(len(labels_plot))
bars_pos = ax.barh([l for l, s in zip(labels_plot, scores_pos) if s > 0],
                    [s for s in scores_pos if s > 0],
                    color='#1D9E75', alpha=0.85, label='Positivo')
bars_neg = ax.barh([l for l, s in zip(labels_plot, scores_neg) if s > 0],
                    [s for s in scores_neg if s > 0],
                    color='#D85A30', alpha=0.85, label='Negativo')
ax.set_xlabel('Score de confiança', fontsize=10)
ax.set_title('Análise de Sentimento com DistilBERT', fontweight='bold')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---
### 3.2 Fill-Mask — BERT Adivinhando Palavras

In [ ]:
# ─────────────────────────────────────────────────────────────
# FILL-MASK — O BERT adivinha palavras mascaradas
# Isso é exatamente o que o BERT faz no pré-treinamento MLM!
#
# [MASK] → posição onde o BERT vai adivinhar a palavra
# top_k   → quantas sugestões retornar
# ─────────────────────────────────────────────────────────────

fill_mask = pipeline('fill-mask', model='bert-base-uncased')

# Frases com [MASK] em posições estratégicas
frases_mascaradas = [
    'Natural language [MASK] allows computers to understand text.',
    'The [MASK] barked at the mailman.',
    'Paris is the [MASK] of France.',
    'Python is a popular programming [MASK].',
]

print('🎭 BERT Fill-Mask — Adivinhando palavras mascaradas\n')

for frase in frases_mascaradas:
    resultados_mask = fill_mask(frase, top_k=5)
    print(f'📝 "{frase}"')
    print(f'   Top 5 previsões do BERT:')
    for i, res in enumerate(resultados_mask):
        barra = '█' * int(res['score'] * 30)
        print(f'   {i+1}. "{res["token_str"]:<20}" {res["score"]:.2%}  {barra}')
    print()

---
### 3.3 Question Answering com BERT

In [ ]:
# ─────────────────────────────────────────────────────────────
# QUESTION ANSWERING (QA)
# Dado um CONTEXTO e uma PERGUNTA, o BERT localiza a resposta
# diretamente no texto — sem gerar nova informação!
# ─────────────────────────────────────────────────────────────

qa_pipeline = pipeline('question-answering',
                        model='distilbert-base-cased-distilled-squad')

# Contextos e perguntas
contextos_qa = [
    {
        'context': (
            'BERT was developed by researchers at Google AI Language. '
            'It was published in October 2018 by Jacob Devlin and colleagues. '
            'BERT achieved state-of-the-art results on 11 NLP benchmarks. '
            'The model was pre-trained on the BooksCorpus and English Wikipedia.'
        ),
        'perguntas': [
            'Who developed BERT?',
            'When was BERT published?',
            'What datasets was BERT trained on?',
        ]
    },
    {
        'context': (
            'Python is a high-level, general-purpose programming language. '
            'It was created by Guido van Rossum and released in 1991. '
            'Python emphasizes code readability and simplicity. '
            'It is widely used in data science, web development, and AI.'
        ),
        'perguntas': [
            'Who created Python?',
            'When was Python released?',
            'What is Python used for?',
        ]
    },
]

print('❓ Question Answering com DistilBERT\n')

for i, item in enumerate(contextos_qa):
    print(f'📄 Contexto {i+1}:')
    print(f'   "{item["context"][:80]}..."\n')
    for pergunta in item['perguntas']:
        resultado = qa_pipeline(
            question=pergunta,
            context=item['context']
        )
        print(f'   ❓ {pergunta}')
        print(f'   ✅ Resposta: "{resultado["answer"]}"  '
              f'(confiança: {resultado["score"]:.2%})')
    print()

---
## 📖 Parte 4 — Família de Modelos BERT

### Comparação dos principais modelos baseados em BERT

In [ ]:
# ─────────────────────────────────────────────────────────────
# COMPARAÇÃO DA FAMÍLIA BERT
# Informações sobre os principais modelos baseados em BERT
# ─────────────────────────────────────────────────────────────

familia_bert = [
    {'nome': 'BERT-base',    'params': 110, 'camadas': 12, 'dims': 768,  'speedup': 1.0,  'cor': '#534AB7'},
    {'nome': 'BERT-large',   'params': 340, 'camadas': 24, 'dims': 1024, 'speedup': 0.4,  'cor': '#3C3489'},
    {'nome': 'DistilBERT',   'params': 66,  'camadas': 6,  'dims': 768,  'speedup': 2.0,  'cor': '#1D9E75'},
    {'nome': 'RoBERTa-base', 'params': 125, 'camadas': 12, 'dims': 768,  'speedup': 1.0,  'cor': '#BA7517'},
    {'nome': 'ALBERT-base',  'params': 12,  'camadas': 12, 'dims': 128,  'speedup': 1.7,  'cor': '#D85A30'},
    {'nome': 'BERTimbau',    'params': 110, 'camadas': 12, 'dims': 768,  'speedup': 1.0,  'cor': '#378ADD'},
]

print('🧠 Família de Modelos Baseados em BERT\n')
print(f'{"Modelo":<16} {"Parâmetros":<14} {"Camadas":<10} {"Dimensão":<12} {"Velocidade"}')
print('-' * 62)
for m in familia_bert:
    destaque = ' ⭐' if m['nome'] == 'DistilBERT' else ''
    print(f'{m["nome"]:<16} {str(m["params"])+"M":<14} {m["camadas"]:<10} '
          f'{m["dims"]:<12} {m["speedup"]:.1f}x{destaque}')

# Gráfico: Parâmetros vs Velocidade
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

nomes  = [m['nome'] for m in familia_bert]
params = [m['params'] for m in familia_bert]
speeds = [m['speedup'] for m in familia_bert]
cores  = [m['cor'] for m in familia_bert]

bars1 = ax1.bar(nomes, params, color=cores, alpha=0.85, edgecolor='white')
ax1.set_title('Parâmetros (Milhões)', fontweight='bold')
ax1.set_ylabel('Parâmetros (M)')
ax1.tick_params(axis='x', rotation=30)
for bar, val in zip(bars1, params):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val}M', ha='center', fontsize=9, fontweight='bold')

bars2 = ax2.bar(nomes, speeds, color=cores, alpha=0.85, edgecolor='white')
ax2.set_title('Velocidade relativa ao BERT-base', fontweight='bold')
ax2.set_ylabel('Speedup (×)')
ax2.tick_params(axis='x', rotation=30)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars2, speeds):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{val}×', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Comparativo da Família BERT', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n⭐ DistilBERT é o mais recomendado para uso em CPU (rápido e compacto)')
print('🇧🇷 BERTimbau é o BERT pré-treinado em Português (recomendado para PT)')

---
## 📖 Parte 5 — BERT em Português

### O BERTimbau foi pré-treinado em 2,7 bilhões de palavras em Português

In [ ]:
# ─────────────────────────────────────────────────────────────
# BERT EM PORTUGUÊS — BERTimbau
# neuralmind/bert-base-portuguese-cased
#
# Treinado em:
#   - brWaC (Brazilian Web as Corpus) — 2.68B palavras
#   - Wikipedia em Português
# ─────────────────────────────────────────────────────────────

print('🇧🇷 Carregando BERTimbau (BERT em Português)...')
print('   (Primeira execução faz download do modelo ~400MB)\n')

tok_pt = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

# Demonstrar tokenização em Português
frases_pt = [
    'O processamento de linguagem natural é fascinante.',
    'A inteligência artificial está transformando o mercado.',
    'Redes neurais aprendem padrões a partir de dados.',
]

print('📝 Tokenização BERTimbau (Português):\n')
for frase in frases_pt:
    tokens = tok_pt.tokenize(frase)
    ids    = tok_pt.encode(frase)
    print(f'  Frase: "{frase}"')
    print(f'  Tokens ({len(tokens)}): {tokens}')
    print(f'  IDs: {ids[:8]}... [{len(ids)} total]')
    print()

print(f'📚 Vocabulário BERTimbau: {tok_pt.vocab_size:,} tokens')
print(f'   (Inclui tokens específicos do Português com acentos: ã, ç, é, etc.)')

# Verificar tokens especiais em PT
especiais = ['português', 'inteligência', 'comunicação', 'ação', 'ções']
print(f'\n🔤 Tokenização de palavras acentuadas:')
for palavra in especiais:
    tok = tok_pt.tokenize(palavra)
    print(f'   "{palavra}" → {tok}')

---
## 📝 Resumo — Guia Rápido de Referência

| Tarefa | Pipeline | Modelo Recomendado | Idioma |
|---|---|---|---|
| Sentimento | `sentiment-analysis` | `distilbert-base-uncased-finetuned-sst-2` | EN |
| Classificação | `zero-shot-classification` | `facebook/bart-large-mnli` | EN |
| Preenchimento | `fill-mask` | `bert-base-uncased` | EN |
| Perguntas | `question-answering` | `distilbert-base-cased-distilled-squad` | EN |
| Resumo | `summarization` | `facebook/bart-large-cnn` | EN |
| NER | `ner` | `dbmdz/bert-large-cased-finetuned-conll03` | EN |
| Sentimento PT | `sentiment-analysis` | `lxyuan/distilbert-base-multilingual-cased-sentiments-student` | PT |
| Embeddings PT | `feature-extraction` | `neuralmind/bert-base-portuguese-cased` | PT |

---

## 🔜 Próximo Passo

Na **Aula 3** você vai entender como o BERT funciona internamente —  
o mecanismo de atenção, os encoders Transformer e por que a bidirecionalidade é possível.